In [ ]:
import pyvisa
rm = pyvisa.ResourceManager()
print(rm.list_resources())

In [ ]:
import pyvisa

rm = pyvisa.ResourceManager()
# Вывод списка всех найденных устройств
print("Доступные устройства:")
for res in rm.list_resources():
    print(f" - {res}")

In [ ]:
# список COM-port
import serial
import serial.tools.list_ports
import time

print("Доступные порты:")
for port in serial.tools.list_ports.comports():
    print(f"  {port.device}: {port.description}")

In [ ]:
# Подключение мощемера (pm400)
from devices.pm_400.PMDevice import PMDevicePM100D, measure_average_power
pm_device=PMDevicePM100D()
current_pm_power = measure_average_power(pm_device)
print('current pm power =',current_pm_power,'mW')


In [ ]:
# Подключение источника напряжения UT3005
from devices.ut_3005.VoltageDriverUT3005 import VoltageDriverUT3005
ut=VoltageDriverUT3005('COM8')
ut.turn_on()
voltage=4
ut.set_voltage(voltage)
a=ut.get_real_voltage()
print('Real voltage = ',a)


In [ ]:
#Подключение РЧ анализатора
import matplotlib.pyplot as plt
f_start=9e+3
f_stop=6.2e+9
f_center=(f_start + f_stop) / 2

from devices.rsa_device.RF306B import RF306B 
rf_device=RF306B()
rf_device.set_cf(f_center)
frequencies, powers=rf_device.get_rf()
plt.plot(frequencies, powers)
plt.figure(figsize=(16, 9))
plt.show()


In [ ]:
# Подключение драйвера тока лазерного диода (cld1015)
from devices.cdl_1015.CLD1015 import CLD1015
import time
LD = CLD1015()
LD.turn_on_all()
LD.set_current(0.3)


In [ ]:
#Подключение линии задержки
from devices.odl_650.OpticDelayLine import OpticDelayLine
odl = OpticDelayLine('COM11')
odl.initialize()
# odl.set_time_delay(100)
# real_delay=odl.get_time_delay()
# print('real_delay =',real_delay)

In [ ]:
odl.set_time_delay(95)
# real_delay=odl.get_time_delay()
# print('real_delay =',real_delay)

In [4]:
from devices.yokogawa.OSA_Yokogawa_new import OSA_Yokogawa_new

osa = OSA_Yokogawa_new()
# osa.set_start('1500nm')
# osa.set_stop('1650nm')



OSA_Yokogawa_new inited!
YOKOGAWA,AQ6370D,91UA07984,02.04



In [ ]:
import serial
import time
TIMEOUT = 3
COM_PORT = "COM12"
BAUDRATE = 9600
ser = serial.Serial(port=COM_PORT, baudrate=BAUDRATE, bytesize=serial.EIGHTBITS, parity=serial.PARITY_NONE, stopbits=serial.STOPBITS_ONE, timeout=TIMEOUT, xonxoff=False, rtscts=False, dsrdtr=False)
print(f"✅ Порт {COM_PORT} открыт\n")

In [ ]:
import itertools
import os
import time
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
from matplotlib import rcParams
from any_functions import create_data_folder

from write_arrays_to_excel import write_arrays_excel
from create_map_and_save import create_map_and_save


# импорт настроек
from config import *
# ===================
# импорт драйверов
# ===================



from devices.rsa_device.rf_measure_lib import rf_measurement
from devices.rsa_device.RF306B import RF306B
from devices.odl_650.OpticDelayLine import OpticDelayLine
from devices.ut_3005.VoltageDriverUT3005 import VoltageDriverUT3005
from devices.cdl_1015.CLD1015 import CLD1015
from devices.pm_400.PMDevice import PMDevicePM100D, measure_average_power

# ========================
# ИНИЦИАЛИЗАЦИЯ ПРИБОРОВ``
# ========================



rf_device = RF306B()
LD = CLD1015()
LD.turn_on_all()

ut = VoltageDriverUT3005('COM8')
ut.turn_on()

pm_device = PMDevicePM100D()



# ============================================================
# Функция для построение 6 карт для одного напряжения
# ============================================================


def _build_voltage_maps(voltage, main_folder, data_buf):
    """Внутренняя функция: строит 6 RF-карт для заданного напряжения."""
    from pathlib import Path
    import numpy as np
    
    # Фильтруем данные по напряжению
    mask = np.array(data_buf['voltage']) == voltage
    
    # Папка для карт этого напряжения
    plot_subfolder = Path(main_folder) / f"voltage_{voltage}V" / "plots"
    plot_subfolder.mkdir(parents=True, exist_ok=True)
    
    # Данные для осей
    x = np.array(data_buf['current'])[mask]
    y = np.array(data_buf['delay'])[mask]
    
    # 6 комбинаций: 3 спана × 2 параметра (частота/мощность)
    maps_config = [
        (data_buf['rf_freq_max'], 'rf_freq_max', 'Частота, ГГц (max span)'),
        (data_buf['rf_pow_max'],  'rf_pow_max',  'Мощность, дБм (max span)'),
        (data_buf['rf_freq_mid'], 'rf_freq_mid', 'Частота, ГГц (mid span)'),
        (data_buf['rf_pow_mid'],  'rf_pow_mid',  'Мощность, дБм (mid span)'),
        (data_buf['rf_freq_min'], 'rf_freq_min', 'Частота, ГГц (min span)'),
        (data_buf['rf_pow_min'],  'rf_pow_min',  'Мощность, дБм (min span)'),
    ]
    
    for z_raw, fname_suffix, z_label in maps_config:
        z = np.array(z_raw)[mask]
        create_map_and_save(
            x_arr=[x.tolist(), "Ток, мА"],
            y_arr=[y.tolist(), "Задержка, пс"],
            z_arr=[z.tolist(), z_label],
            title=f"U = {voltage} В, {z_label}",
            folder_path=plot_subfolder,
            filename=f"RF_{fname_suffix}",
            show_plot=False  # 👈 True только для отладки
        )
    


def main():
    main_folder = create_data_folder(prefix='laser_with_BTF')
    
    params = itertools.product(VOLTAGES, CURRENTS, DELAYS)

    # === Буфер для сбора данных ===
    collected_data = {
        'voltage': [], 'current': [], 'delay': [],
        'rf_freq_max': [], 'rf_pow_max': [],
        'rf_freq_mid': [], 'rf_pow_mid': [],
        'rf_freq_min': [], 'rf_pow_min': [],
    }
    
    current_voltage_batch = None

    # === Списки для Excel ===
    idx_data, time_data, voltage_data, delay_data = [], [], [], []
    current_data,  pm_avg_data = [], []
    rf_freq_max_data, rf_pow_max_data = [], []
    rf_freq_mid_data, rf_pow_mid_data = [], []
    rf_freq_min_data, rf_pow_min_data = [], []
    

    for idx, (voltage, current, delay) in enumerate(params, 1):

        # === Построение карт при смене напряжения ===
        if current_voltage_batch is not None and voltage != current_voltage_batch:
            _build_voltage_maps(current_voltage_batch, main_folder, collected_data)
        current_voltage_batch = voltage

        folder_name = (
            f"voltage_{voltage}V/current_{current}mA/delay_{delay}ps/")
        full_path = os.path.join(main_folder, folder_name)

        # === НАСТРОЙКА ОБОРУДОВАНИЯ ===
        
        ut.set_voltage(voltage=voltage)
        LD.set_current(current=current)
        COMMAND = f"t{delay}"

        ser.write(f"{COMMAND}\r\n".encode('ascii'))
        print(f"➡️  Отправлено: {COMMAND}")
        # Чтение ответа
        response = ""
        start = time.time()
        
        while time.time() - start < TIMEOUT:
            if ser.in_waiting > 0:
                line = ser.readline().decode('ascii', errors='ignore').strip()
                if line:
                    print(f"⬅️  Получено: {line}")
                    response += line + "\n"
        
        

        # === ИЗМЕРЕНИЯ ===
        pm_avg = measure_average_power(pm_device=pm_device)
        
        

        rf_freq_max, rf_pow_max = rf_measurement(
            rf_device=rf_device, N=NUMBER_RF_MEASURE, folder_path=full_path,
            filename='rf_max_span', rf_rbw=RF_RBW_MAX,
            f_start=RF_F_START_MAX, f_stop=RF_F_STOP_MAX, save_png=True)

        
        rf_freq_mid, rf_pow_mid = rf_measurement(
            rf_device=rf_device, N=NUMBER_RF_MEASURE, folder_path=full_path,
            filename='rf_middle_span', rf_rbw=RF_RBW_MIDDLE,f_span=RF_SPAN_MIDDLE,f_center=rf_freq_max, save_png=True)

        
        rf_freq_min, rf_pow_min = rf_measurement(
            rf_device=rf_device, N=NUMBER_RF_MEASURE, folder_path=full_path,
            filename='rf_min_span', rf_rbw=RF_RBW_MIN,
           f_center=rf_freq_mid, f_span=RF_SPAN_MIN,save_png=True)

        


        # === НАКОПЛЕНИЕ ДАННЫХ ===
        idx_data.append(idx)
        time_data.append(datetime.now())
        voltage_data.append(voltage)
        delay_data.append(delay)
        current_data.append(current)
        pm_avg_data.append(pm_avg)
        rf_freq_max_data.append(rf_freq_max)
        rf_pow_max_data.append(rf_pow_max)
        rf_freq_mid_data.append(rf_freq_mid)
        rf_pow_mid_data.append(rf_pow_mid)
        rf_freq_min_data.append(rf_freq_min)
        rf_pow_min_data.append(rf_pow_min)
        

        # === Заполнение буфера для карт ===
        collected_data['voltage'].append(voltage)
        collected_data['current'].append(current)
        collected_data['delay'].append(delay)
        collected_data['rf_freq_max'].append(rf_freq_max)
        collected_data['rf_pow_max'].append(rf_pow_max)
        collected_data['rf_freq_mid'].append(rf_freq_mid)
        collected_data['rf_pow_mid'].append(rf_pow_mid)
        collected_data['rf_freq_min'].append(rf_freq_min)
        collected_data['rf_pow_min'].append(rf_pow_min)

        

    # === Построение карт для ПОСЛЕДНЕГО напряжения ===
    if current_voltage_batch is not None:
        _build_voltage_maps(current_voltage_batch, main_folder, collected_data)

    
    idx_arr = [idx_data, 'number']
    time_arr = [time_data, 'time']
    voltage_arr = [voltage_data, 'voltage_V']
    delay_arr = [delay_data, 'delay_ps']
    current_arr = [current_data, 'current_mA']
    pm_avg_arr = [pm_avg_data, 'pm_avg_mW']
    rf_freq_max_arr = [rf_freq_max_data, 'rf_peak_freq_max_GHz']
    rf_pow_max_arr = [rf_pow_max_data, 'rf_peak_power_max_dBm']
    rf_freq_mid_arr = [rf_freq_mid_data, 'rf_peak_freq_middle_GHz']
    rf_pow_mid_arr = [rf_pow_mid_data, 'rf_peak_power_middle_dBm']
    rf_freq_min_arr = [rf_freq_min_data, 'rf_peak_freq_min_GHz']
    rf_pow_min_arr = [rf_pow_min_data, 'rf_peak_power_min_dBm']
    

    write_arrays_excel(
        idx_arr, time_arr, voltage_arr, delay_arr,
        current_arr, pm_avg_arr,
        rf_freq_max_arr, rf_pow_max_arr,
        rf_freq_mid_arr, rf_pow_mid_arr,
        rf_freq_min_arr, rf_pow_min_arr,
        folder_path=main_folder, filename='results.xlsx')  
    

if __name__ == "__main__":
    main()
